<a href="https://colab.research.google.com/github/jandorrvfx-code/tennis_analysis/blob/main/training/tennis_ball_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Delete everything

In [1]:
!rm -rf *

# Pip Install

In [2]:
%pip install roboflow
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.2 MB/s eta 0:00:00


# Get Dataset

In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key="pKCYijWJauhnpUWiXlwr")
project = rf.workspace("viren-dhanwani").project("tennis-ball-detection")
version = project.version(6)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tennis-ball-detection-6 in yolov8:: 100%|██████████| 1168/1168 [00:00<00:00, 4911.57it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Train Dataset

In [4]:
!yolo task=detect mode=train model=yolov8n.pt data=tennis-ball-detection-6/data.yaml epochs=50 imgsz=640 batch=16
!yolo export model=runs/detect/train/weights/best.pt format=tfjs

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=tennis-ball-detection-6/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

# Download

In [5]:
from google.colab import files

# The path to your generated TFLite model
# tflite_model_path = 'runs/detect/train/weights/best_saved_model/best_float32.tflite'
tflite_model_path = 'runs/detect/train/weights/best.pt'

# Download the file
try:
    files.download(tflite_model_path)
    print(f"Downloading {tflite_model_path}...")
except Exception as e:
    print(f"An error occurred during download: {e}")
    print("You might need to manually locate and download the file from the 'files' tab on the left.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PyTorch to TensorFlow

In [6]:
from ultralytics import YOLO

# Load the YOLO model
model = YOLO("runs/detect/train/weights/best.pt")

# Export the model to TFLite format
model.export(format="tflite")

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 1.5s, saved as 'runs/detect/train/weights/best.onnx' (11.8 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'runs/detect/train/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output T

'runs/detect/train/weights/best_saved_model/best_float32.tflite'

In [8]:
from google.colab import files

# The path to your generated TFLite model
# tflite_model_path = 'runs/detect/train/weights/best_saved_model/best_float32.tflite'
tflite_model_path = 'runs/detect/train/weights/best_saved_model/best_float32.tflite'

# Download the file
try:
    files.download(tflite_model_path)
    print(f"Downloading {tflite_model_path}...")
except Exception as e:
    print(f"An error occurred during download: {e}")
    print("You might need to manually locate and download the file from the 'files' tab on the left.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>